In [31]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

main_colours = plt.cm.rainbow(np.linspace(0, 1, 5))

custom_params = {
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.markeredgecolor": "#131313",
    "lines.markeredgewidth": 1.25,
    "figure.dpi": 200,
    "text.usetex": True,
    "font.family": "serif",
    "axes.prop_cycle": cycler(color=main_colours),
}

sns.set_theme(context="talk", style="ticks", rc=custom_params)

In [32]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.special import i0
from scipy.integrate import trapezoid

In [33]:
MU_RAW = np.array([[1.0, 0.7142857], [0.7142857, 1.0]])
MU = MU_RAW / MU_RAW[0, 0]
MU11 = float(MU[0, 0])  # 1
MU12 = float(MU[0, 1])
MU22 = float(MU[1, 1])

gamma_11 = MU22 / (MU11 * MU22 - MU12**2)


def phi_1(q, alpha, lambd):
    return alpha * (np.sin(2 * np.pi * q / lambd))


def zposneg(kbt, lambd, alpha):
    q1_th = np.linspace(0, lambd, 1_000_000)
    zpos = trapezoid(np.exp(1 / kbt * phi_1(q1_th, alpha, lambd)), q1_th)
    zneg = trapezoid(np.exp(-1 / kbt * phi_1(q1_th, alpha, lambd)), q1_th)

    return zpos, zneg


# ===========================================================================
# Model and numerical parameters
kbt = 1

ENERGY = 4.0
LAMBDA = 1.0
Y_MAX = 10.0
zpos, zneg = zposneg(1, LAMBDA, ENERGY)
D_lj = (1 * LAMBDA**2) / (gamma_11 * zpos * zneg)
# D_lj = 0.010204794283901605


EPSILON_VALUES = np.logspace(-4, 2, 100)

NX = 512
N_HERMITE = 50

OUTPUT_DIR = Path("kubo_corrected_output")

In [34]:
def phi(x: np.ndarray) -> np.ndarray:
    return ENERGY * (1.0 - np.cos(2.0 * np.pi * x / LAMBDA))


def dphi(x: np.ndarray) -> np.ndarray:
    return ENERGY * (2.0 * np.pi / LAMBDA) * np.sin(2.0 * np.pi * x / LAMBDA)

In [35]:
def periodic_sixth_order_matrices(nx: int):
    """Sixth-order centered periodic finite-difference matrices."""
    dx = LAMBDA / nx

    offsets_1 = np.array([-3, -2, -1, 1, 2, 3])
    coeffs_1 = np.array([-1, 9, -45, 45, -9, 1], dtype=float) / (60.0 * dx)

    offsets_2 = np.array([-3, -2, -1, 0, 1, 2, 3])
    coeffs_2 = np.array([2, -27, 270, -490, 270, -27, 2], dtype=float) / (180.0 * dx**2)

    rows, cols, vals = [], [], []
    for i in range(nx):
        for offset, coeff in zip(offsets_1, coeffs_1):
            rows.append(i)
            cols.append((i + offset) % nx)
            vals.append(coeff)
    d1 = sp.coo_matrix((vals, (rows, cols)), shape=(nx, nx)).tocsr()

    rows, cols, vals = [], [], []
    for i in range(nx):
        for offset, coeff in zip(offsets_2, coeffs_2):
            rows.append(i)
            cols.append((i + offset) % nx)
            vals.append(coeff)
    d2 = sp.coo_matrix((vals, (rows, cols)), shape=(nx, nx)).tocsr()

    return dx, d1, d2

In [36]:
DX, D1, D2 = periodic_sixth_order_matrices(NX)
X = -LAMBDA / 2.0 + np.arange(NX) * DX

DPHI = dphi(X)
PEQ_X = np.exp(-phi(X))

DPHI_DIAG = sp.diags(DPHI)
IDENTITY_X = sp.eye(NX, format="csr")

In [37]:
def normalized_hermite_matrix(y: np.ndarray, n_terms: int) -> np.ndarray:
    h = np.empty((len(y), n_terms), dtype=float)
    h[:, 0] = 1.0

    if n_terms > 1:
        h[:, 1] = y

    for n in range(1, n_terms - 1):
        h[:, n + 1] = (y * h[:, n] - np.sqrt(n) * h[:, n - 1]) / np.sqrt(n + 1.0)

    return h

In [38]:
def solve_response(epsilon: float):
    cross = MU12 / np.sqrt(epsilon)
    fast = MU22 / epsilon

    base_x = MU11 * (D2 - DPHI_DIAG @ D1)
    blocks = [[None] * N_HERMITE for _ in range(N_HERMITE)]

    for n in range(N_HERMITE):
        blocks[n][n] = base_x - fast * n * IDENTITY_X

        if n + 1 < N_HERMITE:
            blocks[n][n + 1] = cross * np.sqrt(n + 1.0) * (D1 - DPHI_DIAG)

        if n > 0:
            blocks[n][n - 1] = -cross * np.sqrt(n) * D1

    operator = sp.bmat(blocks, format="csr")

    source = np.zeros(N_HERMITE * NX)
    source[:NX] = -MU11 * DPHI
    source[NX : 2 * NX] = -cross

    constraint = np.zeros(N_HERMITE * NX)
    constraint[:NX] = PEQ_X * DX

    augmented = sp.bmat(
        [
            [operator, constraint[:, None]],
            [constraint[None, :], None],
        ],
        format="csc",
    )

    solution = spla.spsolve(
        augmented,
        np.concatenate([source, np.array([0.0])]),
    )

    coefficients = solution[:-1].reshape(N_HERMITE, NX)
    lagrange_multiplier = solution[-1]

    residual_vector = (
        operator @ solution[:-1] + constraint * lagrange_multiplier - source
    )
    relative_residual = np.linalg.norm(residual_vector) / np.linalg.norm(source)

    z_x = np.sum(PEQ_X) * DX

    correction_x = np.sum(PEQ_X * MU11 * DPHI * coefficients[0]) * DX / z_x
    correction_y = np.sum(PEQ_X * cross * coefficients[1]) * DX / z_x

    diffusion = MU11 - correction_x - correction_y
    diffusion = diffusion / D_lj - 1

    return (
        coefficients,
        float(diffusion),
        float(correction_x),
        float(correction_y),
        float(relative_residual),
    )

In [39]:
def reconstruct_f(
    coefficients: np.ndarray,
    x_indices: np.ndarray,
    y: np.ndarray,
):
    h = normalized_hermite_matrix(y, coefficients.shape[0])
    w = h @ coefficients[:, x_indices]

    x_plot = X[x_indices]
    peq_plot = np.exp(-phi(x_plot)[None, :] - 0.5 * y[:, None] ** 2)

    return x_plot, peq_plot * w

In [40]:
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("\nmu_raw = G G^T =")
    print(MU_RAW)
    print("\nnormalized mu (mu11=1) =")
    print(MU)
    print("\neigenvalues(normalized mu) =")
    print(np.linalg.eigvalsh(MU))

    x_indices = np.linspace(0, NX - 1, 500, dtype=int)
    y_plot = np.linspace(-Y_MAX, Y_MAX, 401)

    rows = []

    for index, epsilon in enumerate(EPSILON_VALUES, start=1):
        print(f"Solving epsilon={epsilon:.6g}")

        (
            coefficients,
            diffusion,
            correction_x,
            correction_y,
            residual,
        ) = solve_response(float(epsilon))

        x_plot, f_plot = reconstruct_f(
            coefficients,
            x_indices,
            y_plot,
        )

        maximum = np.max(np.abs(f_plot))
        levels = np.linspace(-maximum, maximum, 41)

        fig, ax = plt.subplots(figsize=(5, 3.5))
        contour = ax.contourf(
            x_plot,
            y_plot,
            f_plot,
            levels=levels,
        )
        fig.colorbar(contour, ax=ax, label=r"$f(x,y)$", cmap="jet")
        ax.set_xlabel(r"$x=q$")
        ax.set_ylabel(r"$y=u$")
        ax.set_title(
            rf"Corrected Kubo response, $\epsilon={epsilon:.3g}$"
            + "\n"
            + rf"$D_q^*={diffusion:.6g}$"
        )
        fig.tight_layout()
        fig.savefig(
            OUTPUT_DIR / f"f_eps_{index:02d}_{epsilon:.6g}.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.close(fig)

        rows.append(
            {
                "epsilon": epsilon,
                "mu11": MU11,
                "mu12": MU12,
                "mu22": MU22,
                "Dq_dimensionless": diffusion,
                "correction_x": correction_x,
                "correction_y": correction_y,
                "relative_residual": residual,
            }
        )

        del coefficients, f_plot
        gc.collect()

    results = pd.DataFrame(rows)
    results.to_csv(
        OUTPUT_DIR / "corrected_kubo_results.csv",
        index=False,
    )

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.loglog(
        results["epsilon"],
        results["Dq_dimensionless"],
        marker="o",
        linestyle="",
        label="Eq.~3.50",
        markersize=6,
    )
    ax.vlines(0.013, 1e-2, 1e1, color="lavender", alpha=0.8, zorder=-1)
    ax.set_xlabel(r"$\epsilon$")
    ax.set_ylabel(r"$\Delta D^*/D$")
    ax.set_xlim(1e-3, 1e1)
    ax.set_ylim(8e-2, 3.0)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "corrected_diffusion_vs_epsilon.pdf",
        transparent=True,
        bbox_inches="tight",
    )
    plt.close(fig)

    print()
    print(results.to_string(index=False))
    print(f"\nFiles written to {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()


mu_raw = G G^T =
[[1.        0.7142857]
 [0.7142857 1.       ]]

normalized mu (mu11=1) =
[[1.        0.7142857]
 [0.7142857 1.       ]]

eigenvalues(normalized mu) =
[0.2857143 1.7142857]
Solving epsilon=0.0001
Solving epsilon=0.000114976
Solving epsilon=0.000132194
Solving epsilon=0.000151991
Solving epsilon=0.000174753
Solving epsilon=0.000200923
Solving epsilon=0.000231013
Solving epsilon=0.000265609
Solving epsilon=0.000305386
Solving epsilon=0.000351119
Solving epsilon=0.000403702
Solving epsilon=0.000464159
Solving epsilon=0.00053367
Solving epsilon=0.000613591
Solving epsilon=0.00070548
Solving epsilon=0.000811131
Solving epsilon=0.000932603
Solving epsilon=0.00107227
Solving epsilon=0.00123285
Solving epsilon=0.00141747
Solving epsilon=0.00162975
Solving epsilon=0.00187382
Solving epsilon=0.00215443
Solving epsilon=0.00247708
Solving epsilon=0.00284804
Solving epsilon=0.00327455
Solving epsilon=0.00376494
Solving epsilon=0.00432876
Solving epsilon=0.00497702
Solving epsilon=0